# Globular Clusters Random Forest Predictor

### Imports

In [1]:
%matplotlib inline
import matplotlib.pyplot as plt
import sys
sys.path.append('..')
from Utilities import custom
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split, learning_curve
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor

from IPython.core.magic import register_cell_magic
@register_cell_magic
def skip(line, cell):
    return

target='Na/Fe'
if target=='O/Fe':
  plots_folder='../Plots/Performance_Plots/Random_Forest/OxygenPoints'
elif target=='Na/Fe':
  plots_folder='../Plots/Performance_Plots/Random_Forest/SodiumPoints'

### Load/Prep Data

In [2]:
df=custom.data_loader(target)

Some of our data seems to be horizontally displaced (dimmer but with similar distr of abundances). For now we will cut these points out based on filter magnitudes:

In [3]:
cutoffs={'F275W_abs':8.5, 'F336W_abs':6., 'F438W_abs': 6., 'F606W_abs': 4., 'F814W_abs': 3.}

masks=[df[key]>value for key, value in cutoffs.items()]
remove=np.logical_or.reduce(masks)
df=df[~remove]

Let's do some basic processing. want to use one values as magnitude, others as color. Start trying 438 since it is in middle

In [4]:
initial_columns=['F275W_abs', 'F336W_abs', 'F438W_abs', 'F606W_abs', 'F814W_abs']
X_columns=[]

for i in range(len(initial_columns)-1):
    column0, column1=initial_columns[i], initial_columns[i+1]
    new_column=f'{column0}-{column1}'
    X_columns.append(new_column)
    df[new_column]=df[column0]-df[column1]
X_columns.extend(['F438W_abs', 'age_Kruijssen', 'Fe/H'])
#X_columns.extend(['F438W_abs'])
print(X_columns)

['F275W_abs-F336W_abs', 'F336W_abs-F438W_abs', 'F438W_abs-F606W_abs', 'F606W_abs-F814W_abs', 'F438W_abs', 'age_Kruijssen', 'Fe/H']


### Filter NaNs

"Because trees handle missing values natively there is no need for an imputer" CHECK HOW THIS IS DONE

Still need to filter out y-values with NaNs. SOme methods also need both x and y NaNs filtered, so we will make both datasets here:

In [5]:
df.dropna(subset=target, inplace=True)

Eventually want something better to preserve data as much as possible; maybe train to predict based on existing abundances? For now just take rows out when necessary.

# Try a Model

Lets try a very naive random forest:

In [6]:
randregress=RandomForestRegressor(
    n_estimators=10000,
    oob_score=True,
    max_depth=5,
    random_state=42, 
    n_jobs=-1)

X, y=df[X_columns], df[target]
randregress.fit(X, y)

RandomForestRegressor(max_depth=5, n_estimators=10000, n_jobs=-1,
                      oob_score=True, random_state=42)

In [7]:
randregress.oob_score_

0.2599506553904145

In [8]:
X

,F275W_abs-F336W_abs,F336W_abs-F438W_abs,F438W_abs-F606W_abs,F606W_abs-F814W_abs,F438W_abs,age_Kruijssen,Fe/H
Star ID,,,,,,,
R0011193,3.2266,1.5003,1.6079,0.9815,0.812527,12.52,-0.787
R0015269,2.3369,0.9106,1.3140,0.8300,1.733327,12.52,-0.771
R0018788,2.6964,1.0734,1.6548,0.8973,1.507627,12.52,-0.730
R0018794,3.3320,1.7093,1.7314,1.0189,0.456127,12.52,-0.779
R0024168,2.2659,0.5998,1.2333,0.8011,2.275927,12.52,-0.658
...,...,...,...,...,...,...,...
R0002687,1.1538,0.0585,1.0019,0.7334,1.454162,13.06,-2.376
R0002313,1.9211,0.6568,1.3215,0.8733,-0.165738,13.06,NaN
R0004243,2.2142,0.7450,1.3549,0.9038,-0.357038,13.06,-2.370


Wow! Not very good. This is actually significantly worse than the previous fit with less data (both dimensionally and data points). Why/how can this happen?

# Try a NEW Model

GradientBoost is not multivariate; you can use a wrapper to train it on multible variables "at the same time" but it is still not true multivariate, just several single variates. Also cannot have NaNs in the x values.

In [9]:
df_dropna=df.dropna(subset=X_columns)
X, y=df_dropna[X_columns], df_dropna[target]

In [10]:
X

,F275W_abs-F336W_abs,F336W_abs-F438W_abs,F438W_abs-F606W_abs,F606W_abs-F814W_abs,F438W_abs,age_Kruijssen,Fe/H
Star ID,,,,,,,
R0011193,3.2266,1.5003,1.6079,0.9815,0.812527,12.52,-0.787
R0015269,2.3369,0.9106,1.3140,0.8300,1.733327,12.52,-0.771
R0018788,2.6964,1.0734,1.6548,0.8973,1.507627,12.52,-0.730
R0018794,3.3320,1.7093,1.7314,1.0189,0.456127,12.52,-0.779
R0024168,2.2659,0.5998,1.2333,0.8011,2.275927,12.52,-0.658
...,...,...,...,...,...,...,...
R0001196,1.9127,0.5617,1.2856,0.8541,-0.028938,13.06,-2.405
R0002687,1.1538,0.0585,1.0019,0.7334,1.454162,13.06,-2.376
R0004243,2.2142,0.7450,1.3549,0.9038,-0.357038,13.06,-2.370


In [11]:
gradientboost=GradientBoostingRegressor(
    max_depth=5, learning_rate=0.05, n_estimators=10000, 
    subsample=0.6,  
    n_iter_no_change=10, random_state=42)

In [12]:
gradientboost.fit(X, y)

GradientBoostingRegressor(learning_rate=0.05, max_depth=5, n_estimators=10000,
                          n_iter_no_change=10, random_state=42, subsample=0.6)

In [13]:
gradientboost.oob_score_

np.float64(0.005929982040916182)

Oh my God even worse

In [14]:
gradientboost.n_estimators_

81

# Modified Random Forest Regression

By default, each *classifier* tree uses sqrt(N) features available, which for us is only like 2. But it does make the model more robust, and we can always add more trees, since our current optimum is like 50. 

In [15]:
randregress_sqrt=RandomForestRegressor(
    n_estimators=10000,
    oob_score=True,
    max_depth=5,
    max_features='sqrt',
    random_state=42, 
    n_jobs=-1)

X, y=df[X_columns], df[target]
randregress_sqrt.fit(X, y)

RandomForestRegressor(max_depth=5, max_features='sqrt', n_estimators=10000,
                      n_jobs=-1, oob_score=True, random_state=42)

In [16]:
randregress_sqrt.oob_score_

0.20136106739875237

This USED to perform better than when using full features; it now performs worse. 

# Learning Curves

Check how each model progresses as it gets more data

In [17]:
plot_folder='../Plots/Performance_Plots/Random_Forest'
randregress=RandomForestRegressor(
    n_estimators=10000,
    oob_score=True,
    max_depth=5,
    random_state=42, 
    n_jobs=-1)

randregress_sqrt=RandomForestRegressor(
    n_estimators=10000,
    oob_score=True,
    max_depth=5,
    max_features='sqrt',
    random_state=42, 
    n_jobs=-1)


gradientboost=GradientBoostingRegressor(
    max_depth=5, learning_rate=0.05, n_estimators=10000, 
    subsample=0.6,  
    n_iter_no_change=10, random_state=42)

models=[randregress, randregress_sqrt, gradientboost]
labels=['full features', 'sqrt features', 'gradient boost']
colors=['blue', 'orange', 'red']

In [18]:
train_errors_list=[]
valid_errors_list=[]
Xs=[X, X, df_dropna[X_columns]] 
ys=[y, y, df_dropna[target]] 
for i in range(3):
    train_sizes, train_scores, valid_scores = learning_curve(
        models[i], Xs[i], ys[i], cv=5, scoring="neg_root_mean_squared_error"
    )
    train_errors = -train_scores.mean(axis=1)
    valid_errors = -valid_scores.mean(axis=1)
    train_errors_list.append(train_errors)
    valid_errors_list.append(valid_errors)
    print(i)

0
1
2


In [19]:
plt.clf()
plt.ylabel('RMSE')
for i in range(3):
    plt.plot(train_sizes, train_errors_list[i], color=colors[i], linestyle=':', linewidth=2, label=labels[i])
    plt.plot(train_sizes, valid_errors_list[i], color=colors[i], linewidth=3, label=labels[i])
plt.legend()
plt.savefig(f'{plot_folder}/LearningCurve2.png')

# Predictions

In [20]:
for i in range(3):
    models[i].fit(Xs[i], ys[i])

y_pred=[pd.DataFrame(models[i].predict(Xs[i]), columns=[target], index=ys[i].index) for i in range(3)]

In [21]:
#lims=[(-2, 0), (-1, 1), (-0.5, 1)]
target='Na/Fe'
plt.clf()
fig, ax=plt.subplots(1, 3, figsize=(20, 20))
fig.suptitle(target)
for i in range(3):
    ax[i].set_title(labels[i])
    ax[i].set_xlabel('actual')
    ax[i].set_ylabel('pred')
    ax[i].plot(ys[i], ys[i], color='black')
    ax[i].scatter(ys[i], y_pred[i], s=2.0)
    #ax[i].set_xlim(*lims[i])
    #ax[i].set_ylim(*lims[i])
    ax[i].set_aspect('equal')
plt.savefig(f'{plot_folder}/test_pred.png')
#plt.show()


In [23]:
for i in range(3):
    plt.clf()
    plt.hist(y_pred[i], bins=20)
    plt.savefig(f'{plot_folder}/predicted_values_{labels[i]}.png')
    
    plt.clf()
    plt.hist(ys[i], bins=20)
    plt.savefig(f'{plot_folder}/actual_values_{labels[i]}.png')

Alright, this is a bit odd.....the RMSE value for gradient boost is better than the RMSE values for other forests, but the $R^2$ values is significantly worse.... how?

It seems like $R^2$ is not a good metric for non-linear data. IN particular, decisiion trees can get good RMSE with bad $R^2$ on non-linear data. It can also be less useful on smaller datasets, where the variance is more affected by individual points. 

So.....for now, don't worry about it too much?

### Adding Imputation

Let's see what happens if we impute our X and y values to get more data, then re-train our models? The non-gradient models supposedly do not need this, but I am suspicious. And the gradient model does need it.

In [24]:
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import SimpleImputer, KNNImputer, IterativeImputer

multivariate or not? NN best option regardless because truly multivariate, if we can get enough datapoints

Replace with heatmaps?